# Notebook Sidebar — Generic Multi-Utility Shell

**Spec / design** · `2026-06-03` · status: **approved design, pre-plan**
**Subject:** `crates/spur-notebook/jute-notebook/src/ui/notebook`
**Author:** Truong Cong Hoan Vu

Refactor the monolithic right-rail `DatasourceSidebar` into a generic, collapsible **sidebar shell** that hosts multiple **utility panels** using a VS Code activity-bar model. **Datasources becomes the first panel**; an **AI/chat panel** is the anticipated next one (not built here). The shell makes adding a panel cheap — one registry entry plus a body component — without forcing the chat decision today.

## 1. Problem — shell and content are fused

Today `DatasourceSidebar.tsx` (~990 lines) is the *entire* right rail. It owns both concerns at once:

- **Shell responsibilities** — the `collapsed` `useState`, the collapsed icon-rail `<aside>`, the expanded `<aside>`, the header chrome, and the collapse chevrons.
- **Content responsibilities** — datasource attach/detach, four Tauri/daemon `useEffect` listeners (`datasources://changed`, `connections://changed`, `notebook://open_rest_wizard`, global drag-drop), the group input, Add-file / Add-API buttons, the file dropzone, error display, the grouped-entry list, the `SavedConnectionsSection`, and the `AddRestApiWizard` modal.

`NotebookView` simply renders `<DatasourceSidebar />` as the right column of a two-column grid.

**Consequence:** there is no seam between "the rail" and "what's in the rail". Adding any second utility (e.g. AI/chat) would mean forking the whole component or bolting a second concern onto an already-large file.

```mermaid
flowchart LR
  NV["NotebookView<br/>grid: [ cells | sidebar ]"] --> DS["DatasourceSidebar<br/><b>shell + content fused</b>"]
  DS --> Chrome["collapse state · aside wrappers · header chevrons"]
  DS --> Content["datasource logic · 4 listeners · dropzone<br/>saved connections · AddRestApiWizard"]
  classDef bad fill:#fee,stroke:#c00,color:#900;
  class DS bad;
```

## 2. Goals & non-goals

**Goals**
- Separate a reusable **shell** (collapsible rail + icon activity bar) from **content panels**.
- Make datasources *one* panel among several; adding a panel = one registry entry + a body component.
- Provide **programmatic activation** so a utility can make itself the active panel (the existing `open_rest_wizard` flow needs this; chat will too).
- Preserve all current datasource behavior and its tests.

**Non-goals (YAGNI for now)**
- Persisting active-panel / collapsed state across sessions — **ephemeral session state** only.
- Building the AI/chat panel — only making it cheap to add later.
- Per-panel availability predicates, panel-supplied header toolbars, or a runtime `register()` API.
- Drop-to-attach while a non-datasource panel is the *visible* panel (see §9).

## 3. Decisions

| # | Question | Decision | Rationale |
|---|----------|----------|-----------|
| A | Driver | Concrete next utility (AI/chat) **and** general extensibility | User wants both |
| B | Presentation | **Activity-bar + single active panel** (VS Code style) | Datasources & chat are both tall / content-heavy; one-at-a-time fits; an icon rail scales |
| C | Rail placement | Far-right edge (outermost); panel to its left | Sidebar is right-aligned — the mirror of VS Code's left rail |
| D | State location | New ephemeral zustand store `useSidebar` (no `persist`) | Mirrors `stores/settings.ts`; programmatic activation needs shared state |
| E | Persistence | **Ephemeral session state** | YAGNI; trivial to add `persist` later |
| F | Activation | `activatePanel(id)` sets active **and** forces expand | `open_rest_wizard` and future chat need a "focus me" hook |
| G | Registry | Static module array `SIDEBAR_PANELS` | Simplest; no runtime registration needed |
| H | Mounting | **Keep-alive**: default panel mounts immediately and stays mounted; inactive panels hidden via `hidden`, not unmounted | Keeps every panel's listeners live so `open_rest_wizard` fires regardless of which panel is visible (see §7) |

## 4. Target architecture

```mermaid
flowchart RL
  NV["NotebookView<br/>grid: [ cells | sidebar ]"] --> NS["NotebookSidebar<br/><b>shell</b>"]

  subgraph Shell["NotebookSidebar (shell)"]
    direction TB
    Rail["activity rail<br/>always visible · far right<br/>icon per panel + collapse toggle"]
    Header["panel header (active title)"]
    Body["panel region<br/>(all panels mounted; inactive hidden)"]
  end

  NS --> Rail
  NS --> Header
  NS --> Body

  Store["useSidebar store<br/>{ activePanelId, collapsed }<br/>activatePanel · toggleCollapsed · setCollapsed"]
  Reg["SIDEBAR_PANELS registry<br/>[ { id, title, icon, ariaLabel, Component } ]"]

  Rail -. reads/sets .-> Store
  Body -. reads .-> Store
  NS -. iterates .-> Reg
  Body --> DP["DatasourcePanel<br/>(content only)"]
  Reg -. id = chat (future) .-> CP["ChatPanel<br/>(future)"]
  classDef future fill:#eef,stroke:#88a,stroke-dasharray:4 3,color:#446;
  class CP future;
```

- The rail renders one icon button per `SIDEBAR_PANELS` entry plus the collapse toggle; the active icon is highlighted.
- The panel region (header + bodies) is hidden when `collapsed`, leaving only the rail.
- `NotebookView`'s grid is **unchanged** — it renders `<NotebookSidebar />` instead of `<DatasourceSidebar />`.

## 5. Shell behavior — collapse & activation

```mermaid
stateDiagram-v2
  [*] --> Expanded
  Expanded --> Collapsed: toggleCollapsed()
  Collapsed --> Expanded: toggleCollapsed()
  Collapsed --> Expanded: click rail icon / activatePanel(id)
  Expanded --> Expanded: click another rail icon (swap active panel)
  note right of Collapsed
    icon rail still visible
    panel region width = 0
  end note
```

- **Collapsed:** only the icon rail shows. Clicking any panel icon **expands and activates** that panel (`activatePanel` clears `collapsed`).
- **Expanded:** clicking a different icon swaps the active panel; clicking the collapse toggle hides the panel region.
- The collapsed-rail `DatabaseIcon` block that `DatasourceSidebar` renders today is replaced by the shell's generic rail — datasources contributes its icon via its registry entry.

## 6. State store — `src/stores/sidebar.ts`

Mirrors the existing `stores/settings.ts` zustand pattern, **without** the `persist` middleware (ephemeral).

```ts
type SidebarState = {
  activePanelId: string;
  collapsed: boolean;
};

type SidebarActions = {
  activatePanel: (id: string) => void;   // sets id AND collapsed = false
  toggleCollapsed: () => void;
  setCollapsed: (collapsed: boolean) => void;
};

// defaults: activePanelId = SIDEBAR_PANELS[0].id, collapsed = false
export const useSidebar = create<SidebarState & SidebarActions>()((set) => ({ ... }));
```

Because it is a module-level singleton (like `useSettings`), any code can drive it imperatively via `useSidebar.getState().activatePanel("datasources")` — including non-React event listeners and tests.

## 7. Mounting strategy & listener liveness (the load-bearing decision)

A naive activity bar renders **only the active panel**, unmounting the rest. That breaks a real flow: `DatasourcePanel` registers the `notebook://open_rest_wizard` Tauri listener inside a `useEffect`. If that listener is unmounted while the user is on another panel (e.g. chat), the event fires into the void and the REST wizard never opens.

**Decision (H): keep-alive mounting.** The panel region renders the default panel immediately and keeps every activated panel **mounted**, merely hiding inactive ones with the `hidden` attribute. Datasources is the default panel, so it is mounted from first render and its four listeners stay live for the whole session — identical to today's behavior.

```mermaid
flowchart TB
  subgraph Region["panel region (all mounted)"]
    DPm["DatasourcePanel<br/>visible · listeners live"]
    CPm["ChatPanel (future)<br/>hidden · listeners live"]
  end
  Region -. active=datasources .-> DPm
  Region -. hidden .-> CPm
  Note["Inactive panel = hidden attribute,<br/>NOT unmounted → useEffect listeners persist"]
  Region --- Note
  classDef note fill:#ffd,stroke:#cc0,color:#660;
  class Note note;
```

**Programmatic activation flow** — the REST-wizard event focuses datasources, then opens the wizard:

```mermaid
sequenceDiagram
  participant Evt as Tauri event<br/>notebook://open_rest_wizard
  participant DP as DatasourcePanel listener<br/>(always mounted)
  participant Store as useSidebar
  participant Shell as NotebookSidebar
  Evt->>DP: payload (prefill)
  DP->>Store: activatePanel("datasources")
  Store-->>Shell: activePanelId = datasources, collapsed = false
  DP->>DP: setRestWizardPrefill(...) + open wizard
  Note over Shell,DP: datasource panel becomes visible; wizard opens
```

**Cost / trade-off:** every registered panel's effects run even while hidden. For datasources this is unchanged from today. For a future chat panel this is usually *desirable* (keep the connection / stream alive in the background). If a panel ever proves too heavy to keep mounted, we can switch *that* panel to lazy-unmount and hoist only its cross-cutting listeners to the shell — but we do not need that now.

## 8. Panel interface & registry

A panel is described by a small descriptor — this is the entire extension surface.

```ts
// src/ui/notebook/sidebar/types.ts
import type { LucideIcon } from "lucide-react";

export type SidebarPanel = {
  id: string;                       // stable key, e.g. "datasources"
  title: string;                    // header label, e.g. "Datasources"
  icon: LucideIcon;                 // rail icon, e.g. DatabaseIcon
  ariaLabel: string;                // rail button aria-label
  Component: React.ComponentType;   // panel body — NO shell chrome
};
```

```ts
// src/ui/notebook/sidebar/panels.ts
import { DatabaseIcon } from "lucide-react";
import DatasourcePanel from "./DatasourcePanel";

export const SIDEBAR_PANELS: SidebarPanel[] = [
  {
    id: "datasources",
    title: "Datasources",
    icon: DatabaseIcon,
    ariaLabel: "Datasources",
    Component: DatasourcePanel,
  },
];
```

- The store's default `activePanelId` is `SIDEBAR_PANELS[0].id`.
- The rail maps over `SIDEBAR_PANELS` to render icon buttons; the panel region renders each panel's `Component`, hiding all but the active one.
- **Adding the AI/chat panel later** = write `ChatPanel.tsx` + append one entry to this array. No shell changes. Open it programmatically with `useSidebar.getState().activatePanel("chat")`.

**Deliberate YAGNI** (call out if any should be pulled in now): no per-panel availability predicate, no panel-supplied header toolbar slot (each panel renders its own controls in its body, as datasources does today), and a static array rather than a runtime registration API.

## 9. Refactor — `DatasourceSidebar` → `DatasourcePanel`

The current component is split so the datasource logic sheds its shell responsibilities.

**Moves to the shell (deleted from the datasource component):**
- The `collapsed` `useState` and both the collapsed and expanded `<aside>` wrappers.
- The collapse/expand buttons (`ChevronLeftIcon` / `ChevronRightIcon`) and the standalone collapsed-rail `DatabaseIcon`.
- The outer header row (`DatabaseIcon` + "Datasources" title + collapse chevron). The title now comes from the shell's panel header; the `DatabaseIcon` becomes the rail icon via the registry entry.

**Stays, as `DatasourcePanel` (the body, no shell):**
- All daemon logic, the four `useEffect` listeners, the group input, Add / API buttons, the dropzone, error display, the grouped-entry list, `SavedConnectionsSection`, and the `AddRestApiWizard` modal.
- The `open_rest_wizard` listener gains **one line**: call `useSidebar.getState().activatePanel("datasources")` before opening the wizard (works because the panel is kept mounted — §7).
- All pure helpers (`groupDatasourceEntries`, `upsertDatasourceEntry`, `datasourceNameFromPath`, `normalizeGroup`, `firstSelectedPath`, `firstDroppedPath`, `isDatasourcePath`, `isPositionInsideElement`, `errorMessage`) and sub-components (`DatasourceListItem`, `DatasourceKindBadge`, `DatasourceColumnRow`, `SavedConnectionsSection`, `SavedConnectionRow`) move unchanged. `restWizardPrefillFromPayload` stays exported (tests import it).

**File organization** — a focused `sidebar/` subfolder:

```mermaid
flowchart TB
  subgraph FS["src/ui/notebook/sidebar/"]
    NSf["NotebookSidebar.tsx<br/>shell: rail + collapse + panel header/region"]
    Pf["panels.ts<br/>SIDEBAR_PANELS registry"]
    Tf["types.ts<br/>SidebarPanel type"]
    DPf["DatasourcePanel.tsx<br/>extracted body (was DatasourceSidebar)"]
    Wf["AddRestApiWizard.tsx<br/>moved here (datasource-specific)"]
  end
  Store["src/stores/sidebar.ts<br/>useSidebar"]
  NV["src/ui/notebook/NotebookView.tsx<br/>renders <NotebookSidebar/>"]
  NSf --> Pf --> DPf --> Wf
  NSf --> Tf
  NSf -.-> Store
  DPf -.-> Store
  NV --> NSf
```

## 10. Dropzone caveat (accepted)

The file dropzone and the global Tauri `onDragDropEvent` handler live inside `DatasourcePanel` and hit-test the drop position against the dropzone's own ref via `isPositionInsideElement`. Under keep-alive mounting the dropzone DOM is mounted whenever datasources has been activated, **but it is `hidden` when another panel is the visible one**, so `getBoundingClientRect()` yields an empty rect and drops won't register while a different panel is showing.

**Accepted for now:** drop-to-attach is a datasource-specific affordance; requiring datasources to be the visible panel to drop a file is reasonable. If we later want "drop anywhere → auto-switch to datasources", we hoist the drag-drop handler to the shell and have it call `activatePanel("datasources")` on a datasource-typed drop. Out of scope here.

## 11. Testing

**Migrated** — `DatasourceSidebar.test.tsx` → `DatasourcePanel.test.tsx`:
- The ~15 existing tests assert datasource behavior only (attach, detach, saved connections expand/attach/delete, multi-table render, api-tables render, `restWizardPrefillFromPayload`). **None** assert collapse/shell chrome, so they migrate near-verbatim.
- Swap import + `render(<DatasourceSidebar />)` → `render(<DatasourcePanel />)`. All daemon/Tauri mocks unchanged.
- Extend the `open_rest_wizard` test to also assert `useSidebar.getState().activePanelId === "datasources"` after the event (reset the store in `beforeEach`).

**New tests:**
- `NotebookSidebar.test.tsx` (shell): renders one rail button per `SIDEBAR_PANELS` entry; clicking an icon activates its panel; the collapse toggle hides the panel region but keeps the rail; activating while collapsed expands; inactive panels are `hidden` but present in the DOM (guards keep-alive mounting).
- `stores/__tests__/sidebar.test.ts`: `activatePanel` sets id **and** clears `collapsed`; `toggleCollapsed` flips; `setCollapsed` sets.

## 12. Build sequence

Each step compiles and tests green before the next; datasources works end-to-end after step 5 with no intermediate broken state.

```mermaid
flowchart LR
  S1["1 · stores/sidebar.ts<br/>+ store test"] --> S2["2 · sidebar/types.ts"]
  S2 --> S3["3 · extract DatasourcePanel<br/>strip chrome · add activatePanel<br/>move AddRestApiWizard · migrate test"]
  S3 --> S4["4 · NotebookSidebar.tsx<br/>+ panels.ts + shell test"]
  S4 --> S5["5 · NotebookView → NotebookSidebar<br/>delete DatasourceSidebar.tsx"]
```

**Definition of done:** `pnpm test` green in `jute-notebook`; `pnpm lint` / typecheck clean; the right rail behaves exactly as before (collapse, attach via picker/drop, API wizard, saved connections), now served through the generic shell; adding a second panel is demonstrably one registry entry.

---
*Status: approved design, pre-plan. Next step: writing-plans → implementation plan (beads DAG).*